# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 30
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import functions as F

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 30 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 30
Session ID: 89517685-934d-4a73-bba2-a72505ed148d
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 89517685-934d-4a73-bba2-a72505ed148d to get into ready status...
Session 89517685-934d-4a73-bba2-a72505ed148d has 

In [2]:
dyf = glueContext.create_dynamic_frame.from_catalog(database="flowlogs_db", table_name="raw_awslogs")
print(dyf.count())
dyf.printSchema()

8274
root
|-- version: long
|-- account-id: long
|-- interface-id: string
|-- srcaddr: string
|-- dstaddr: string
|-- srcport: long
|-- dstport: long
|-- protocol: long
|-- packets: long
|-- bytes: long
|-- start: long
|-- end: long
|-- action: string
|-- log-status: string
|-- vpc-id: string
|-- subnet-id: string
|-- instance-id: string
|-- tcp-flags: long
|-- flow-direction: string
|-- pkt-srcaddr: string
|-- pkt-dstaddr: string
|-- aws-account-id: string
|-- aws-service: string
|-- aws-region: string
|-- year: string
|-- month: string
|-- day: string
|-- hour: string


In [3]:
nums = ["version","srcport","dstport","protocol","packets","bytes","start","end","tcp-flags"]
dyf2 = dyf.resolveChoice(specs=[(c, "cast:long") for c in nums] + [("account-id", "cast:string")])
dyf2.printSchema()

root
|-- version: long
|-- account-id: string
|-- interface-id: string
|-- srcaddr: string
|-- dstaddr: string
|-- srcport: long
|-- dstport: long
|-- protocol: long
|-- packets: long
|-- bytes: long
|-- start: long
|-- end: long
|-- action: string
|-- log-status: string
|-- vpc-id: string
|-- subnet-id: string
|-- instance-id: string
|-- tcp-flags: long
|-- flow-direction: string
|-- pkt-srcaddr: string
|-- pkt-dstaddr: string
|-- aws-account-id: string
|-- aws-service: string
|-- aws-region: string
|-- year: string
|-- month: string
|-- day: string
|-- hour: string


In [4]:
df = dyf2.toDF()
df.cache()
df.show(5, truncate=False)

+-------+------------+---------------------+-----------+-----------+-------+-------+--------+-------+-----+----------+----------+------+----------+---------------------+------------------------+-------------------+---------+--------------+-----------+-----------+--------------+-----------+----------+----+-----+---+----+
|version|account-id  |interface-id         |srcaddr    |dstaddr    |srcport|dstport|protocol|packets|bytes|start     |end       |action|log-status|vpc-id               |subnet-id               |instance-id        |tcp-flags|flow-direction|pkt-srcaddr|pkt-dstaddr|aws-account-id|aws-service|aws-region|year|month|day|hour|
+-------+------------+---------------------+-----------+-----------+-------+-------+--------+-------+-----+----------+----------+------+----------+---------------------+------------------------+-------------------+---------+--------------+-----------+-----------+--------------+-----------+----------+----+-----+---+----+
|5      |209901331329|eni-046426ca

In [5]:
df.groupBy("log-status").count().show()

+----------+-----+
|log-status|count|
+----------+-----+
|        OK| 8274|
+----------+-----+


In [ ]:
df.filter(F.col("dstport").isNull()).groupBy("log-status").count().show()

In [6]:
(df.filter(F.col("action") == "REJECT")
   .groupBy("dstport").count()
   .orderBy(F.desc("count"))
   .show(10))

+-------+-----+
|dstport|count|
+-------+-----+
|     23|   50|
|    443|   50|
|      0|   41|
|     22|   39|
|   3389|   21|
|   8080|   20|
|     80|   20|
|   8443|   19|
|   5060|   16|
|  30718|   16|
+-------+-----+
only showing top 10 rows


In [8]:
df.select("start", F.col("start").cast("timestamp").alias("start_time")).show(5)

+----------+-------------------+
|     start|         start_time|
+----------+-------------------+
|1794945580|2026-11-17 19:59:40|
|1794945580|2026-11-17 19:59:40|
|1794945580|2026-11-17 19:59:40|
|1794945580|2026-11-17 19:59:40|
|1794945580|2026-11-17 19:59:40|
+----------+-------------------+
only showing top 5 rows


In [9]:
df2 = df.toDF(*[c.replace("-", "_") for c in df.columns])
df2.printSchema()

root
 |-- version: long (nullable = true)
 |-- account_id: string (nullable = true)
 |-- interface_id: string (nullable = true)
 |-- srcaddr: string (nullable = true)
 |-- dstaddr: string (nullable = true)
 |-- srcport: long (nullable = true)
 |-- dstport: long (nullable = true)
 |-- protocol: long (nullable = true)
 |-- packets: long (nullable = true)
 |-- bytes: long (nullable = true)
 |-- start: long (nullable = true)
 |-- end: long (nullable = true)
 |-- action: string (nullable = true)
 |-- log_status: string (nullable = true)
 |-- vpc_id: string (nullable = true)
 |-- subnet_id: string (nullable = true)
 |-- instance_id: string (nullable = true)
 |-- tcp_flags: long (nullable = true)
 |-- flow_direction: string (nullable = true)
 |-- pkt_srcaddr: string (nullable = true)
 |-- pkt_dstaddr: string (nullable = true)
 |-- aws_account_id: string (nullable = true)
 |-- aws_service: string (nullable = true)
 |-- aws_region: string (nullable = true)
 |-- year: string (nullable = true)
 |

In [10]:
df3 = df2.withColumn(
    "src_subnet_24",
    F.concat(F.regexp_extract("srcaddr", r"^(\d+\.\d+\.\d+)\.", 1), F.lit(".0/24"))
)
(df3.filter(F.col("action") == "REJECT")
    .groupBy("src_subnet_24")
    .agg(F.countDistinct("srcaddr").alias("source_ips"),
         F.countDistinct("dstport").alias("ports_probed"))
    .orderBy(F.desc("ports_probed"))
    .show(10, truncate=False))

+----------------+----------+------------+
|src_subnet_24   |source_ips|ports_probed|
+----------------+----------+------------+
|45.141.233.0/24 |10        |352         |
|35.203.211.0/24 |177       |289         |
|162.216.150.0/24|174       |276         |
|85.217.149.0/24 |54        |268         |
|162.216.149.0/24|154       |256         |
|35.203.210.0/24 |157       |228         |
|147.185.133.0/24|148       |209         |
|10.50.1.0/24    |1         |199         |
|147.185.132.0/24|115       |162         |
|204.76.203.0/24 |5         |81          |
+----------------+----------+------------+
only showing top 10 rows


In [ ]:
%stop_session